In [ ]:
%cd ..

In [ ]:

# import ipympl
import matplotlib
%matplotlib widget
from pymultipact.domain import Project
from pymultipact.domain import Domain
import numpy as np

In [ ]:
# create project
proj = Project()
proj.create_project('TESLA')

# create domain
domain = Domain(proj)

# define elliptical cavity geometry
# mid_cell = np.array([62.22, 66.1261, 30.22022, 23.1131, 71.9869, 93.5, 171.1929])*1e-3
# domain.define_elliptical_cavity(mid_cell=mid_cell)

domain.draw_mesh()

In [ ]:
domain.compute_fields()

In [ ]:
domain.draw_fields(1, 'E')

In [ ]:
domain.mesh

In [ ]:
# Create an empty list to store your data
data = []

# Loop through all the mesh points
for v in domain.mesh.vertices:
    z, r = v.point
    
    # Get electric field values (er, ez) at point p
    e_field = domain.gfu_E[1](domain.mesh(z, r))
    ez, er = e_field[0], e_field[1]  # Assuming the first two components represent (er, ez)

    # Get magnetic field values (hr, hz) at point p
    h_field = domain.gfu_H[1](domain.mesh(z, r))
    ht = h_field[0].imag  # Assuming the first two components represent (hr, hz)
    

    # Append (r, z, er, ez, hr, hz) to the list
    data.append([z, r, ez, er, ht])
print(type(ht)) 
# Convert the list to a NumPy array
data_array = np.array(data)

In [ ]:
data_array

In [ ]:
# Apply the query using boolean masking with array indexing
import time
start = time.time()
z_query, r_query = [-5.76524000e-02,  0.00000000e+00]
mask = (data_array[:, 0] == z_query) & (data_array[:, 1] == r_query)
# Return the last three entries (columns 2, 3, 4, 5) for the matching rows
result = data_array[mask][:, 2:]

print(result[0][-1], (time.time()-start)*1e6, 'us')

In [ ]:
query_array = np.array([
    [-5.76e-02,  0.00000000e+00],
    [-5.76524000e-02,  3.50000000e-02],
    [-5.76205060e-02,  3.50000670e-02]])

# Separate r and z from the query array
r_query = query_array[:, 0]
z_query = query_array[:, 1]

# Compare all r and z values from the data array with the queries
# Broadcasting: this creates a 2D boolean array where each row is compared against the entire r_query and z_query
r_matches = (data_array[:, 0, np.newaxis] == r_query)
z_matches = (data_array[:, 1, np.newaxis] == z_query)

# Logical AND to get rows where both r and z match
matches = r_matches & z_matches

# Extract the rows from data_array where the (r, z) pairs match
# Using `np.any` to check if any column in matches is True (i.e., at least one match for each row)
matched_rows = data_array[np.any(matches, axis=1), 2:]
print(matched_rows)

In [ ]:
# field and phase range
epks = np.linspace(4.30890052356021, 90, 179) * 1e6
phis = np.linspace(0, 2 * np.pi, 72)

# check initial points in range
xrange = [-0.025, 0.0]
domain.show_initial_points(xrange, step=0.002)

In [ ]:
# begin multipacting analysis
# domain.analyse_multipacting(epks=epks, phis=phis, xrange=xrange, step=0.002)
domain.analyse_multipacting(epks=epks, phis=phis, xrange=xrange, step=0.002, proc_count=12)

In [ ]:
# domain.plot_sey()

In [ ]:
Ef = domain.calculate_Ef()

In [ ]:
domain.plot_Ef()

In [ ]:
domain.plot_cf()

In [ ]:
domain.plot_ef()

In [ ]:
domain.plot_trajectories()